# GoEmotions Exploration

This notebook is for practicing a data-quality interview workflow on a human-labeled dataset.

Goals:
- Load the raw GoEmotions annotation shards.
- Inspect label columns, annotation volume, and possible duplicates.
- Create starter metrics for annotator coverage, agreement, and label distributions.
- Keep the work reproducible enough to discuss during an interview.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "full_dataset"
BERT_DIR = ROOT / "models" / "bert"

DATA_DIR.mkdir(parents=True, exist_ok=True)
BERT_DIR.mkdir(parents=True, exist_ok=True)

ROOT, DATA_DIR, BERT_DIR

## Download Data

The original Google Research repository uses `wget`. This notebook uses Python so it works on macOS environments where `wget` may not be installed.

In [ ]:
# Original commands from the Google Research repo:
# wget -P data/full_dataset/ https://storage.googleapis.com/gresearch/goemotions/data/full_dataset/goemotions_1.csv
# wget -P data/full_dataset/ https://storage.googleapis.com/gresearch/goemotions/data/full_dataset/goemotions_2.csv
# wget -P data/full_dataset/ https://storage.googleapis.com/gresearch/goemotions/data/full_dataset/goemotions_3.csv

from urllib.request import urlretrieve

urls = [
    "https://storage.googleapis.com/gresearch/goemotions/data/full_dataset/goemotions_1.csv",
    "https://storage.googleapis.com/gresearch/goemotions/data/full_dataset/goemotions_2.csv",
    "https://storage.googleapis.com/gresearch/goemotions/data/full_dataset/goemotions_3.csv",
]

for url in urls:
    target = DATA_DIR / url.rsplit("/", 1)[-1]
    if not target.exists():
        print(f"Downloading {target.name}")
        urlretrieve(url, target)
    else:
        print(f"Already present: {target.name}")

## BERT Repo / Model Assets

The BERT code repository is cloned into `models/bert/google-research-bert`. The historical Google Cloud Storage BERT zip URLs linked from the archived repo may return `AccessDenied`. If that happens, use Hugging Face's `google-bert/bert-base-uncased` files as a practical fallback.

In [ ]:
# Clone the BERT code repository if needed.
!test -d ../models/bert/google-research-bert || git clone https://github.com/google-research/bert ../models/bert/google-research-bert

# Original Google Research zip URL from the archived README. This may currently return AccessDenied.
# !curl -L https://storage.googleapis.com/bert_models/2018_10_18/uncased_L-12_H-768_A-12.zip -o ../models/bert/uncased_L-12_H-768_A-12.zip
# !unzip -o ../models/bert/uncased_L-12_H-768_A-12.zip -d ../models/bert/

# Hugging Face fallback examples:
# !mkdir -p ../models/bert/bert-base-uncased
# !curl -L https://huggingface.co/google-bert/bert-base-uncased/resolve/main/config.json -o ../models/bert/bert-base-uncased/config.json
# !curl -L https://huggingface.co/google-bert/bert-base-uncased/resolve/main/vocab.txt -o ../models/bert/bert-base-uncased/vocab.txt
# !curl -L https://huggingface.co/google-bert/bert-base-uncased/resolve/main/model.safetensors -o ../models/bert/bert-base-uncased/model.safetensors

## Load Raw Annotation Shards

In [ ]:
csv_paths = sorted(DATA_DIR.glob("goemotions_*.csv"))
csv_paths

In [ ]:
raw = pd.concat((pd.read_csv(path) for path in csv_paths), ignore_index=True)
raw.shape

In [ ]:
raw.head()

In [ ]:
raw.info()

## Basic Data Checks

In [ ]:
summary = {
    "rows": len(raw),
    "columns": raw.shape[1],
    "unique_text": raw["text"].nunique() if "text" in raw else np.nan,
    "unique_ids": raw["id"].nunique() if "id" in raw else np.nan,
    "duplicate_rows": raw.duplicated().sum(),
}
summary

In [ ]:
raw.isna().sum().sort_values(ascending=False).head(20)

## Label Columns

GoEmotions stores emotion labels as one-hot columns. The metadata columns below are excluded from the label list.

In [ ]:
metadata_cols = {
    "text", "id", "author", "subreddit", "link_id", "parent_id", "created_utc",
    "rater_id", "example_very_unclear"
}
label_cols = [col for col in raw.columns if col not in metadata_cols]
label_cols, len(label_cols)

In [ ]:
label_counts = raw[label_cols].sum().sort_values(ascending=False)
label_counts

## Annotator Coverage

The full GoEmotions CSVs include one row per rater annotation, so `rater_id` is the annotator identifier.

In [ ]:
annotator_counts = raw["rater_id"].value_counts().rename_axis("rater_id").reset_index(name="num_annotations")
annotator_counts.describe()

In [ ]:
annotations_per_example = raw.groupby("id")["rater_id"].nunique()
annotations_per_example.describe()

## Agreement Starter

For a multi-label dataset, agreement is not just one majority class. This starter converts each row into a tuple of selected labels, then measures the most common label-set share per example.

In [ ]:
def selected_labels(row):
    return tuple(label for label in label_cols if row[label] == 1)

raw["label_set"] = raw.apply(selected_labels, axis=1)

agreement = (
    raw.groupby("id")["label_set"]
    .agg(num_annotations="size", top_label_set_count=lambda s: s.value_counts().iloc[0])
    .reset_index()
)
agreement["percentage_agreement"] = agreement["top_label_set_count"] / agreement["num_annotations"]
agreement.describe()

In [ ]:
raw.merge(agreement, on="id").sort_values("percentage_agreement").head(20)[
    ["id", "text", "rater_id", "label_set", "num_annotations", "percentage_agreement"]
]

## Interview Prep Next Steps

- Build a consensus label per example.
- Estimate rater reliability using peer agreement.
- Compare majority vote vs weighted vote.
- Inspect examples where weighting changes the final label.
- Decide whether to exclude unclear examples or treat `example_very_unclear` as a quality signal.